In [1]:
import os
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

project_root = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "analysisVR").is_dir() and (p / "baseVR").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find project root containing analysisVR and baseVR")

sys.path.append(str(project_root))
from baseVR.base_functionality import init_import_paths

init_import_paths()

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from analytics_processing.sessions_from_nas_parsing import (
    fullfnames2snames,
    sessionlist_fullfnames_from_args,
)
from CustomLogger import CustomLogger as Logger

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 60)
px.defaults.template = "plotly_white"

Logger().init_logger(None, None, logging_level="WARNING")


# Choice / Cue / Outcome Encoding With Coarse-to-Fine Single-Neuron GLMs

This notebook quantifies how single-neuron firing relates to task features in event-aligned trial windows, moving from a coarse population view to finer session-wise and trial-wise tuning. The analysis uses analytics already defined in this repository: `FiringRate40msHz`, `Behavior40msAligned`, `TrialWiseT0Events40ms`, and optional `SpikeClusterMetadata` for region labels.

**Core approach**
- Build trial-level event windows from `TrialWiseT0Events40ms`.
- Convert 40 ms firing rates back to canonical spike counts with `count = firing_rate_hz / 25`.
- Aggregate counts and behavioral controls inside each event window for every `session_id x unit x trial_id x interval_name`.
- Fit count GLMs with a log link and `log(n_bins_present)` exposure offset.
- Compare Poisson and Negative Binomial families by 5-fold held-out log-likelihood, then keep the better family per neuron-session-interval fit.
- Quantify cue, choice, and outcome encoding with reduced-model drops in held-out fraction deviance explained and full-vs-reduced likelihood-ratio tests, followed by BH-FDR correction.

**Main intervals**
- `cue_entry_interval`
- `cue_exit_interval`
- `R1_entry_interval`
- `R2_entry_interval`
- `reward1_sound_interval`
- `reward2_sound_interval`

**Model predictors**
- Task features: `cue_binary`, `choice_side_binary`, `outcome_binary`
- Controls: `trial_number_z`, `speed_mean`, `rotation_mean`, `lick_rate`

**Implementation details**
- Count GLMs are fit with `statsmodels`.
- Plotting is done with `plotly` only.
- Trial-wise refinement uses interaction models (`feature x trial_number_z`) and early/middle/late within-session summaries for the strongest global interval-feature pairs.


In [2]:
ANIMAL_IDS = [6]
PARADIGM_IDS = [1100]
SESSION_IDS = None
EXCL_SESSION_NAMES = [
    "2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min",
    "2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min",
    "2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min",
]

MAIN_INTERVALS = [
    "cue_entry_interval",
    "cue_exit_interval",
    "R1_entry_interval",
    "R2_entry_interval",
    "reward1_sound_interval",
    "reward2_sound_interval",
]

INTERVAL_LABELS = {
    "cue_entry_interval": "Cue entry",
    "cue_exit_interval": "Cue exit",
    "R1_entry_interval": "R1 entry",
    "R2_entry_interval": "R2 entry",
    "reward1_sound_interval": "Reward 1 sound",
    "reward2_sound_interval": "Reward 2 sound",
}

FEATURE_SPECS = {
    "cue": {"column": "cue_binary", "label0": "Cue 1", "label1": "Cue 2"},
    "choice": {"column": "choice_side_binary", "label0": "R1", "label1": "R2"},
    "outcome": {"column": "outcome_binary", "label0": "Unrewarded", "label1": "Rewarded"},
}

CONTROL_SOURCE_COLS = {
    "speed_mean": "frame_raw_500msMedian",
    "rotation_mean": "frame_YawPitch_abs_vel_sum_500msMedian",
    "lick_rate": "lick_detected",
}

REQUIRED_BEHAVIOR_COLS = [
    "trial_id",
    "cue",
    "trial_outcome",
    "choice_R1",
    "choice_R2",
    "from_ephys_timestamp",
    "to_ephys_timestamp",
    *CONTROL_SOURCE_COLS.values(),
]

PRIMARY_FEATURES = list(FEATURE_SPECS.keys())
FULL_PREDICTOR_COLS = [
    "cue_binary",
    "choice_side_binary",
    "outcome_binary",
    "trial_number_z",
    "speed_mean",
    "rotation_mean",
    "lick_rate",
]
CONTINUOUS_PREDICTORS = ["trial_number_z", "speed_mean", "rotation_mean", "lick_rate"]

MIN_TRIALS_PER_GROUP = 18
N_CV_FOLDS = 5
RANDOM_STATE = 42
FDR_ALPHA = 0.05
TOP_TRIALWISE_PAIRS_PER_FEATURE = 1
MIN_PHASE_TRIALS = 8
ROLLING_WINDOW_MIN = 5
PROGRESS_EVERY = 250
MAX_GROUPS = None


def _ensure_flat(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()
    return out


def _parse_interval(value):
    if isinstance(value, pd.Interval):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, dict):
        left = value.get("left")
        right = value.get("right")
        if pd.notna(left) and pd.notna(right):
            return pd.Interval(float(left), float(right), closed="right")
        return np.nan
    if isinstance(value, (tuple, list)) and len(value) >= 2:
        left, right = value[0], value[1]
        if pd.notna(left) and pd.notna(right):
            return pd.Interval(float(left), float(right), closed="right")
        return np.nan
    txt = str(value).strip()
    if txt.lower() in {"", "nan", "none", "nat"}:
        return np.nan
    match = re.match(r"^[\(\[]\s*([-+0-9eE\.]+)\s*,\s*([-+0-9eE\.]+)\s*[\)\]]$", txt)
    if match:
        return pd.Interval(float(match.group(1)), float(match.group(2)), closed="right")
    return np.nan


def _normalize_unit_from_series(series: pd.Series) -> pd.Series:
    txt = series.astype(str).str.strip()
    from_label = txt.str.extract(r"(?i)unit\s*0*(\d+)", expand=False)
    from_label_num = pd.to_numeric(from_label, errors="coerce")
    from_num = pd.to_numeric(series, errors="coerce")
    unit_num = from_label_num.combine_first(from_num)
    return unit_num.map(lambda v: f"Unit{int(v):04d}" if pd.notna(v) else np.nan)


def _build_meta_map_from_spike(spike_df: pd.DataFrame | None) -> pd.DataFrame:
    if spike_df is None or len(spike_df) == 0:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    mdf = _ensure_flat(spike_df)
    if "session_id" not in mdf.columns:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    area_candidates = ["fine_brain_area", "brain_area", "region", "area"]
    area_col = next((c for c in area_candidates if c in mdf.columns), None)
    if area_col is None:
        mdf["brain_region"] = "Unknown"
    else:
        mdf["brain_region"] = (
            mdf[area_col]
            .astype(str)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "None": np.nan})
            .fillna("Unknown")
        )

    unit_col = None
    for candidate in ["unit", "unit_id", "unit_name", "cluster_id", "entry_id"]:
        if candidate in mdf.columns:
            unit_series = _normalize_unit_from_series(mdf[candidate])
            if unit_series.notna().any():
                mdf["unit"] = unit_series
                unit_col = candidate
                break

    if "unit" not in mdf.columns:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    if unit_col == "entry_id":
        shifted = _normalize_unit_from_series(pd.to_numeric(mdf["entry_id"], errors="coerce") + 1)
        if shifted.notna().sum() > mdf["unit"].notna().sum():
            mdf["unit"] = shifted

    meta_map = mdf[["session_id", "unit", "brain_region"]].dropna(subset=["session_id", "unit"]).copy()
    meta_map = (
        meta_map.groupby(["session_id", "unit"], as_index=False)["brain_region"]
        .agg(lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0])
    )
    return meta_map


def _prepare_trial_numbers(base_df: pd.DataFrame) -> pd.DataFrame:
    trial_order = (
        base_df[["session_id", "trial_id"]]
        .drop_duplicates()
        .sort_values(["session_id", "trial_id"])
        .reset_index(drop=True)
    )
    trial_order["trial_number"] = trial_order.groupby("session_id").cumcount() + 1
    base_df = base_df.merge(trial_order, on=["session_id", "trial_id"], how="left")

    def _zscore_trial_order(series: pd.Series) -> pd.Series:
        std = series.std(ddof=0)
        if pd.isna(std) or std == 0:
            return pd.Series(np.zeros(len(series), dtype=float), index=series.index)
        return (series - series.mean()) / std

    base_df["trial_number_z"] = (
        base_df.groupby("session_id", group_keys=False)["trial_number"].apply(_zscore_trial_order)
    )
    return base_df


def build_unit_window_table(
    fr_df: pd.DataFrame,
    behavior_df: pd.DataFrame,
    t0_long: pd.DataFrame,
    unit_cols: list[str],
    meta_map: pd.DataFrame | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    fr_df = fr_df.sort_values(["session_id", "from_ephys_timestamp", "to_ephys_timestamp"]).copy()
    behavior_df = behavior_df.sort_values(["session_id", "from_ephys_timestamp", "to_ephys_timestamp"]).copy()
    t0_long = t0_long.sort_values(["session_id", "trial_id", "interval_name"]).copy()

    base_rows = []
    spike_rows = []

    session_ids = sorted(set(fr_df["session_id"]) & set(behavior_df["session_id"]) & set(t0_long["session_id"]))
    control_cols = list(CONTROL_SOURCE_COLS.keys())

    for session_id in session_ids:
        fr_s = fr_df.loc[fr_df["session_id"] == session_id].copy()
        beh_s = behavior_df.loc[behavior_df["session_id"] == session_id].copy()
        evt_s = t0_long.loc[t0_long["session_id"] == session_id].copy()

        if fr_s.empty or beh_s.empty or evt_s.empty:
            continue

        fr_mid = fr_s["bin_mid"].to_numpy(dtype=float)
        beh_mid = beh_s["bin_mid"].to_numpy(dtype=float)

        count_mat = fr_s[unit_cols].to_numpy(dtype=np.int32)
        cum_counts = np.vstack(
            [np.zeros((1, count_mat.shape[1]), dtype=np.int64), np.cumsum(count_mat, axis=0, dtype=np.int64)]
        )

        beh_mat = beh_s[control_cols].to_numpy(dtype=float)
        beh_valid = np.isfinite(beh_mat).astype(np.int32)
        beh_sum = np.nan_to_num(beh_mat, nan=0.0)
        cum_beh_sum = np.vstack(
            [np.zeros((1, beh_mat.shape[1]), dtype=float), np.cumsum(beh_sum, axis=0, dtype=float)]
        )
        cum_beh_n = np.vstack(
            [np.zeros((1, beh_mat.shape[1]), dtype=np.int32), np.cumsum(beh_valid, axis=0, dtype=np.int32)]
        )

        for row in evt_s.itertuples(index=False):
            interval = row.interval
            if not isinstance(interval, pd.Interval):
                continue

            lo_fr = int(np.searchsorted(fr_mid, float(interval.left), side="left"))
            hi_fr = int(np.searchsorted(fr_mid, float(interval.right), side="right"))
            if hi_fr <= lo_fr:
                continue

            lo_beh = int(np.searchsorted(beh_mid, float(interval.left), side="left"))
            hi_beh = int(np.searchsorted(beh_mid, float(interval.right), side="right"))

            ctrl_mean = np.full(len(control_cols), np.nan, dtype=float)
            if hi_beh > lo_beh:
                seg_sum = cum_beh_sum[hi_beh] - cum_beh_sum[lo_beh]
                seg_n = cum_beh_n[hi_beh] - cum_beh_n[lo_beh]
                np.divide(seg_sum, seg_n, out=ctrl_mean, where=seg_n > 0)

            base_rows.append(
                {
                    "session_id": row.session_id,
                    "trial_id": row.trial_id,
                    "interval_name": row.interval_name,
                    "cue": row.cue,
                    "trial_outcome": row.trial_outcome,
                    "choice_R1": row.choice_R1,
                    "choice_R2": row.choice_R2,
                    "n_bins_present": hi_fr - lo_fr,
                    **{control_cols[i]: ctrl_mean[i] for i in range(len(control_cols))},
                }
            )
            spike_rows.append(cum_counts[hi_fr] - cum_counts[lo_fr])

    if not base_rows:
        raise ValueError("No interval-aligned rows were built from the loaded analytics.")

    base_df = pd.DataFrame.from_records(base_rows)
    base_df = _prepare_trial_numbers(base_df)

    base_df["cue"] = pd.to_numeric(base_df["cue"], errors="coerce")
    base_df["choice_R1"] = pd.to_numeric(base_df["choice_R1"], errors="coerce")
    base_df["choice_R2"] = pd.to_numeric(base_df["choice_R2"], errors="coerce")

    base_df["cue_binary"] = np.where(base_df["cue"] == 1, 0.0, np.where(base_df["cue"] == 2, 1.0, np.nan))
    base_df["choice_side_binary"] = np.where(
        (base_df["choice_R1"] == 1) & (base_df["choice_R2"] != 1),
        0.0,
        np.where((base_df["choice_R2"] == 1) & (base_df["choice_R1"] != 1), 1.0, np.nan),
    )
    base_df["outcome_binary"] = (
        ((base_df["cue"] == 1) & (base_df["choice_R1"] == 1))
        | ((base_df["cue"] == 2) & (base_df["choice_R2"].isin([1, 2])))
    ).astype(float)

    count_matrix = np.vstack(spike_rows).astype(np.int32)
    repeated = base_df.loc[base_df.index.repeat(len(unit_cols))].copy().reset_index(drop=True)
    repeated["unit"] = np.tile(unit_cols, len(base_df))
    repeated["spike_count"] = count_matrix.reshape(-1).astype(np.int32)

    if meta_map is not None and not meta_map.empty:
        repeated = repeated.merge(meta_map, on=["session_id", "unit"], how="left")
    else:
        repeated["brain_region"] = "Unknown"

    repeated["brain_region"] = repeated["brain_region"].fillna("Unknown")
    repeated["interval_label"] = repeated["interval_name"].map(INTERVAL_LABELS).fillna(repeated["interval_name"])

    design_required = ["cue_binary", "choice_side_binary", "outcome_binary", "trial_number_z", "n_bins_present"]
    repeated = repeated.dropna(subset=design_required).copy()
    repeated = repeated[repeated["n_bins_present"] > 0].copy()

    fit_keys = ["session_id", "unit", "interval_name"]
    eligibility = repeated.groupby(fit_keys).agg(
        n_trials=("trial_id", "nunique"),
        count_var=("spike_count", "var"),
    )
    eligible_keys = eligibility[
        (eligibility["n_trials"] >= MIN_TRIALS_PER_GROUP) & eligibility["count_var"].fillna(0).gt(0)
    ].reset_index()[fit_keys]

    repeated = repeated.merge(eligible_keys, on=fit_keys, how="inner")
    repeated["fit_id"] = (
        repeated["session_id"].astype(str)
        + " | "
        + repeated["unit"].astype(str)
        + " | "
        + repeated["interval_name"].astype(str)
    )
    return repeated.reset_index(drop=True), base_df


In [3]:
session_dirs, _ = sessionlist_fullfnames_from_args(
    PARADIGM_IDS,
    ANIMAL_IDS,
    SESSION_IDS,
    excl_session_names=EXCL_SESSION_NAMES,
)
session_names = fullfnames2snames(session_dirs)

fr_raw = analytics.get_analytics("FiringRate40msHz", session_names=session_names)
behavior_raw = analytics.get_analytics(
    "Behavior40msAligned",
    session_names=session_names,
    columns=REQUIRED_BEHAVIOR_COLS,
)
t0_events_raw = analytics.get_analytics("TrialWiseT0Events40ms", session_names=session_names)

try:
    spike_meta_raw = analytics.get_analytics("SpikeClusterMetadata", session_names=session_names)
except Exception:
    spike_meta_raw = None

fr_flat = _ensure_flat(fr_raw)
behavior_flat = _ensure_flat(behavior_raw)
t0_flat = _ensure_flat(t0_events_raw)
meta_map = _build_meta_map_from_spike(spike_meta_raw)

unit_cols = [c for c in fr_flat.columns if str(c).startswith("Unit")]
fr_flat["bin_mid"] = (
    pd.to_numeric(fr_flat["from_ephys_timestamp"], errors="coerce")
    + pd.to_numeric(fr_flat["to_ephys_timestamp"], errors="coerce")
) / 2.0
fr_flat[unit_cols] = np.rint(fr_flat[unit_cols].fillna(0).to_numpy(dtype=float) / 25.0).astype(np.int32)

behavior_flat["bin_mid"] = (
    pd.to_numeric(behavior_flat["from_ephys_timestamp"], errors="coerce")
    + pd.to_numeric(behavior_flat["to_ephys_timestamp"], errors="coerce")
) / 2.0
behavior_flat = behavior_flat.rename(columns={v: k for k, v in CONTROL_SOURCE_COLS.items()})

interval_cols_found = [name for name in MAIN_INTERVALS if name in t0_flat.columns]
if len(interval_cols_found) != len(MAIN_INTERVALS):
    missing_intervals = sorted(set(MAIN_INTERVALS) - set(interval_cols_found))
    raise ValueError(f"Missing expected interval columns in TrialWiseT0Events40ms: {missing_intervals}")

for name in interval_cols_found:
    t0_flat[name] = t0_flat[name].map(_parse_interval)

t0_long = (
    t0_flat[["session_id", "trial_id", "cue", "trial_outcome", "choice_R1", "choice_R2", *interval_cols_found]]
    .melt(
        id_vars=["session_id", "trial_id", "cue", "trial_outcome", "choice_R1", "choice_R2"],
        value_vars=interval_cols_found,
        var_name="interval_name",
        value_name="interval",
    )
    .dropna(subset=["interval"])
    .copy()
)
t0_long["interval_label"] = t0_long["interval_name"].map(INTERVAL_LABELS).fillna(t0_long["interval_name"])

unit_window_table, interval_base_table = build_unit_window_table(
    fr_df=fr_flat,
    behavior_df=behavior_flat,
    t0_long=t0_long,
    unit_cols=unit_cols,
    meta_map=meta_map,
)

print(f"Sessions loaded: {len(session_names)}")
print(f"Units available: {len(unit_cols)}")
print(f"Aligned trial-window rows (interval base): {len(interval_base_table):,}")
print(f"Aligned unit-window rows: {len(unit_window_table):,}")

display(
    unit_window_table[
        [
            "session_id",
            "trial_id",
            "interval_label",
            "unit",
            "spike_count",
            "n_bins_present",
            "cue_binary",
            "choice_side_binary",
            "outcome_binary",
            "trial_number_z",
            "speed_mean",
            "rotation_mean",
            "lick_rate",
            "brain_region",
        ]
    ].head(10)
)


Sessions loaded: 31
Units available: 77
Aligned trial-window rows (interval base): 11,877
Aligned unit-window rows: 375,415


,session_id,trial_id,interval_label,unit,spike_count,n_bins_present,cue_binary,choice_side_binary,outcome_binary,trial_number_z,speed_mean,rotation_mean,lick_rate,brain_region
0,2024-11-14_16-40,3.0,R1 entry,Unit0001,2,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,CA1
1,2024-11-14_16-40,3.0,R1 entry,Unit0002,0,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,CA1
2,2024-11-14_16-40,3.0,R1 entry,Unit0003,0,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,CA1
3,2024-11-14_16-40,3.0,R1 entry,Unit0004,0,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,DG
4,2024-11-14_16-40,3.0,R1 entry,Unit0005,1,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,DG
5,2024-11-14_16-40,3.0,R1 entry,Unit0007,0,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,DG
6,2024-11-14_16-40,3.0,R1 entry,Unit0009,0,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,DG
7,2024-11-14_16-40,3.0,R1 entry,Unit0010,16,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,DG
8,2024-11-14_16-40,3.0,R1 entry,Unit0012,1,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,DG
9,2024-11-14_16-40,3.0,R1 entry,Unit0013,0,40,1.0,0.0,0.0,-1.655942,39.274024,15.002384,0.0,DG


In [4]:
def _estimate_alpha(y: np.ndarray) -> float:
    y = np.asarray(y, dtype=float)
    mu = float(np.mean(y))
    var = float(np.var(y, ddof=1)) if y.size > 1 else 0.0
    if not np.isfinite(mu) or mu <= 0:
        return 1e-8
    if not np.isfinite(var) or var <= mu:
        return 1e-8
    return float(max((var - mu) / (mu ** 2), 1e-8))


def _make_family(family_name: str, alpha: float):
    if family_name == "poisson":
        return sm.families.Poisson()
    return sm.families.NegativeBinomial(alpha=float(max(alpha, 1e-8)))


def _heldout_loglik(y_true: np.ndarray, mu_pred: np.ndarray, family_name: str, alpha: float) -> float:
    mu_pred = np.clip(np.asarray(mu_pred, dtype=float), 1e-9, None)
    y_true = np.asarray(y_true, dtype=int)
    if family_name == "poisson":
        return float(stats.poisson.logpmf(y_true, mu_pred).sum())
    size = 1.0 / float(max(alpha, 1e-8))
    prob = size / (size + mu_pred)
    return float(stats.nbinom.logpmf(y_true, size, prob).sum())


def _heldout_deviance(y_true: np.ndarray, mu_pred: np.ndarray, family_name: str, alpha: float) -> float:
    fam = _make_family(family_name, alpha)
    return float(fam.deviance(np.asarray(y_true, dtype=float), np.clip(np.asarray(mu_pred, dtype=float), 1e-9, None)))


def _build_design(train_df: pd.DataFrame, predictors: list[str], reference_df: pd.DataFrame | None = None):
    if reference_df is None:
        reference_df = train_df

    train_x = train_df[predictors].astype(float).copy()
    ref_x = reference_df[predictors].astype(float).copy()
    scale_info = {}

    for col in predictors:
        if col in CONTINUOUS_PREDICTORS:
            mean = float(train_x[col].mean())
            std = float(train_x[col].std(ddof=0))
            if not np.isfinite(std) or std == 0:
                std = 1.0
            train_x[col] = (train_x[col] - mean) / std
            ref_x[col] = (ref_x[col] - mean) / std
            scale_info[col] = (mean, std)

    train_x = sm.add_constant(train_x, has_constant="add")
    ref_x = sm.add_constant(ref_x, has_constant="add")
    return train_x, ref_x, scale_info


def _fit_glm(train_df: pd.DataFrame, predictors: list[str], family_name: str, ref_df: pd.DataFrame | None = None):
    ref_df = train_df if ref_df is None else ref_df
    train_x, ref_x, _ = _build_design(train_df, predictors, reference_df=ref_df)
    alpha = _estimate_alpha(train_df["spike_count"].to_numpy()) if family_name == "negbin" else 1e-8
    family = _make_family(family_name, alpha)
    offset_train = np.log(np.clip(train_df["n_bins_present"].to_numpy(dtype=float), 1.0, None))
    result = sm.GLM(
        train_df["spike_count"].to_numpy(dtype=float),
        train_x,
        family=family,
        offset=offset_train,
    ).fit(maxiter=200, disp=0)
    return result, ref_x, alpha


def _lr_pvalue(full_result, reduced_result) -> float:
    lr_stat = float(2.0 * (full_result.llf - reduced_result.llf))
    df_diff = float(full_result.df_model - reduced_result.df_model)
    if df_diff <= 0:
        return np.nan
    return float(stats.chi2.sf(max(lr_stat, 0.0), df_diff))


def _available_predictors(group_df: pd.DataFrame) -> list[str]:
    cols = []
    for col in FULL_PREDICTOR_COLS:
        vals = pd.to_numeric(group_df[col], errors="coerce")
        if vals.notna().sum() < 2:
            continue
        if vals.nunique(dropna=True) <= 1:
            continue
        cols.append(col)
    return cols


def _cv_splits(n_rows: int) -> list[tuple[np.ndarray, np.ndarray]]:
    n_splits = int(min(N_CV_FOLDS, n_rows))
    if n_splits < 2:
        return []
    indices = np.arange(n_rows)
    fold_sizes = np.full(n_splits, n_rows // n_splits, dtype=int)
    fold_sizes[: n_rows % n_splits] += 1
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(indices)
    current = 0
    splits = []
    for fold_size in fold_sizes:
        test_idx = shuffled[current : current + fold_size]
        train_mask = np.ones(n_rows, dtype=bool)
        train_mask[test_idx] = False
        train_idx = indices[train_mask]
        current += fold_size
        if len(train_idx) == 0 or len(test_idx) == 0:
            continue
        splits.append((train_idx, test_idx))
    return splits


def fit_single_group(group_df: pd.DataFrame) -> dict | None:
    group_df = group_df.sort_values("trial_number").reset_index(drop=True)
    n_trials = int(group_df["trial_id"].nunique())
    if n_trials < MIN_TRIALS_PER_GROUP:
        return None

    predictors = _available_predictors(group_df)
    if len(predictors) == 0:
        return None

    y = group_df["spike_count"].to_numpy(dtype=int)
    if np.var(y) <= 0:
        return None

    splits = _cv_splits(len(group_df))
    if not splits:
        return None

    family_scores = {}
    full_cv_fde = {}
    reduced_cv_fde = {}

    for family_name in ["poisson", "negbin"]:
        total_ll = 0.0
        total_dev_full = 0.0
        total_dev_null = 0.0
        dev_by_reduced = {feature: 0.0 for feature in PRIMARY_FEATURES}
        valid_reduced = {feature: False for feature in PRIMARY_FEATURES}

        for train_idx, test_idx in splits:
            train_df = group_df.iloc[train_idx].copy()
            test_df = group_df.iloc[test_idx].copy()

            try:
                full_res, test_x_full, alpha = _fit_glm(train_df, predictors, family_name, ref_df=test_df)
                null_res, test_x_null, _ = _fit_glm(train_df, [], family_name, ref_df=test_df)
            except Exception:
                total_ll = np.nan
                total_dev_full = np.nan
                total_dev_null = np.nan
                break

            test_offset = np.log(np.clip(test_df["n_bins_present"].to_numpy(dtype=float), 1.0, None))
            mu_full = np.clip(full_res.predict(test_x_full, offset=test_offset), 1e-9, None)
            mu_null = np.clip(null_res.predict(test_x_null, offset=test_offset), 1e-9, None)
            y_test = test_df["spike_count"].to_numpy(dtype=int)

            total_ll += _heldout_loglik(y_test, mu_full, family_name, alpha)
            total_dev_full += _heldout_deviance(y_test, mu_full, family_name, alpha)
            total_dev_null += _heldout_deviance(y_test, mu_null, family_name, alpha)

            for feature_name, spec in FEATURE_SPECS.items():
                feature_col = spec["column"]
                if feature_col not in predictors:
                    continue
                reduced_predictors = [col for col in predictors if col != feature_col]
                try:
                    reduced_res, test_x_red, _ = _fit_glm(train_df, reduced_predictors, family_name, ref_df=test_df)
                    mu_red = np.clip(reduced_res.predict(test_x_red, offset=test_offset), 1e-9, None)
                    dev_by_reduced[feature_name] += _heldout_deviance(y_test, mu_red, family_name, alpha)
                    valid_reduced[feature_name] = True
                except Exception:
                    continue

        family_scores[family_name] = total_ll
        full_cv_fde[family_name] = np.nan if not np.isfinite(total_dev_null) or total_dev_null <= 0 else 1.0 - (total_dev_full / total_dev_null)
        reduced_cv_fde[family_name] = {}
        for feature_name in PRIMARY_FEATURES:
            if valid_reduced[feature_name] and np.isfinite(total_dev_null) and total_dev_null > 0:
                reduced_cv_fde[family_name][feature_name] = 1.0 - (dev_by_reduced[feature_name] / total_dev_null)
            else:
                reduced_cv_fde[family_name][feature_name] = np.nan

    family_series = pd.Series(family_scores, dtype=float)
    if family_series.notna().sum() == 0:
        return None

    chosen_family = family_series.idxmax()
    chosen_fde_cv = full_cv_fde.get(chosen_family, np.nan)

    try:
        full_res, full_x, alpha = _fit_glm(group_df, predictors, chosen_family, ref_df=group_df)
        null_res, null_x, _ = _fit_glm(group_df, [], chosen_family, ref_df=group_df)
    except Exception:
        return None

    full_offset = np.log(np.clip(group_df["n_bins_present"].to_numpy(dtype=float), 1.0, None))
    mu_full = np.clip(full_res.predict(full_x, offset=full_offset), 1e-9, None)
    mu_null = np.clip(null_res.predict(null_x, offset=full_offset), 1e-9, None)
    dev_full_train = _heldout_deviance(y, mu_full, chosen_family, alpha)
    dev_null_train = _heldout_deviance(y, mu_null, chosen_family, alpha)
    fde_train = np.nan if dev_null_train <= 0 else 1.0 - (dev_full_train / dev_null_train)

    record = {
        "session_id": group_df["session_id"].iat[0],
        "unit": group_df["unit"].iat[0],
        "brain_region": group_df["brain_region"].iat[0] if "brain_region" in group_df.columns else "Unknown",
        "interval_name": group_df["interval_name"].iat[0],
        "interval_label": group_df["interval_label"].iat[0],
        "n_trials": n_trials,
        "mean_count": float(np.mean(y)),
        "var_count": float(np.var(y, ddof=1)) if len(y) > 1 else np.nan,
        "overdispersion_index": float(np.var(y, ddof=1) / np.mean(y)) if np.mean(y) > 0 and len(y) > 1 else np.nan,
        "family_selected": chosen_family,
        "poisson_cv_loglik": family_scores.get("poisson", np.nan),
        "negbin_cv_loglik": family_scores.get("negbin", np.nan),
        "full_fde_cv": chosen_fde_cv,
        "full_fde_train": fde_train,
        "alpha_nb": alpha if chosen_family == "negbin" else np.nan,
    }

    for feature_name, spec in FEATURE_SPECS.items():
        feature_col = spec["column"]
        record[f"{feature_name}_coef"] = full_res.params.get(feature_col, np.nan)
        if feature_col not in predictors:
            record[f"{feature_name}_delta_fde"] = np.nan
            record[f"{feature_name}_p"] = np.nan
            continue

        reduced_predictors = [col for col in predictors if col != feature_col]
        try:
            reduced_res, _, _ = _fit_glm(group_df, reduced_predictors, chosen_family, ref_df=group_df)
            record[f"{feature_name}_p"] = _lr_pvalue(full_res, reduced_res)
        except Exception:
            reduced_res = None
            record[f"{feature_name}_p"] = np.nan

        reduced_fde = reduced_cv_fde.get(chosen_family, {}).get(feature_name, np.nan)
        record[f"{feature_name}_delta_fde"] = (
            chosen_fde_cv - reduced_fde if np.isfinite(chosen_fde_cv) and np.isfinite(reduced_fde) else np.nan
        )

    available_deltas = {feature: record.get(f"{feature}_delta_fde", np.nan) for feature in PRIMARY_FEATURES}
    available_deltas = {k: v for k, v in available_deltas.items() if np.isfinite(v)}
    if available_deltas:
        record["dominant_feature"] = max(available_deltas, key=available_deltas.get)
        record["dominant_delta_fde"] = available_deltas[record["dominant_feature"]]
    else:
        record["dominant_feature"] = np.nan
        record["dominant_delta_fde"] = np.nan
    return record


fit_groups = list(unit_window_table.groupby(["session_id", "unit", "interval_name"], sort=False))
if MAX_GROUPS is not None:
    fit_groups = fit_groups[: int(MAX_GROUPS)]

fit_records = []
for idx, (_, group_df) in enumerate(fit_groups, start=1):
    rec = fit_single_group(group_df)
    if rec is not None:
        fit_records.append(rec)
    if idx == 1 or idx % PROGRESS_EVERY == 0 or idx == len(fit_groups):
        print(f"[global-fit] processed {idx}/{len(fit_groups)} groups | kept={len(fit_records)}")

glm_results = pd.DataFrame(fit_records)
if glm_results.empty:
    raise ValueError("No eligible GLM fits were produced.")

for feature_name in PRIMARY_FEATURES:
    p_col = f"{feature_name}_p"
    q_col = f"{feature_name}_q"
    sig_col = f"{feature_name}_sig"
    glm_results[q_col] = np.nan
    mask = glm_results[p_col].notna()
    if mask.any():
        glm_results.loc[mask, q_col] = multipletests(glm_results.loc[mask, p_col], method="fdr_bh")[1]
    glm_results[sig_col] = glm_results[q_col] < FDR_ALPHA

preferred = []
for row in glm_results.itertuples(index=False):
    candidates = []
    for feature_name in PRIMARY_FEATURES:
        delta = getattr(row, f"{feature_name}_delta_fde")
        sig = getattr(row, f"{feature_name}_sig")
        if pd.notna(delta) and bool(sig):
            candidates.append((feature_name, float(delta)))
    if candidates:
        preferred.append(max(candidates, key=lambda item: item[1])[0])
    else:
        preferred.append("none")
glm_results["preferred_feature"] = preferred

feature_rows = []
for feature_name in PRIMARY_FEATURES:
    feature_rows.append(
        glm_results[
            [
                "session_id",
                "unit",
                "brain_region",
                "interval_name",
                "interval_label",
                "family_selected",
                "full_fde_cv",
                "n_trials",
            ]
        ]
        .assign(
            feature=feature_name,
            delta_fde=glm_results[f"{feature_name}_delta_fde"],
            p_value=glm_results[f"{feature_name}_p"],
            q_value=glm_results[f"{feature_name}_q"],
            is_significant=glm_results[f"{feature_name}_sig"],
            coef=glm_results[f"{feature_name}_coef"],
        )
    )

feature_results = pd.concat(feature_rows, ignore_index=True)

display(glm_results.head(10))


[global-fit] processed 1/7213 groups | kept=1
[global-fit] processed 250/7213 groups | kept=240
[global-fit] processed 500/7213 groups | kept=483
[global-fit] processed 750/7213 groups | kept=716
[global-fit] processed 1000/7213 groups | kept=950
[global-fit] processed 1250/7213 groups | kept=1185
[global-fit] processed 1500/7213 groups | kept=1422
[global-fit] processed 1750/7213 groups | kept=1664
[global-fit] processed 2000/7213 groups | kept=1901
[global-fit] processed 2250/7213 groups | kept=2096
[global-fit] processed 2500/7213 groups | kept=2326
[global-fit] processed 2750/7213 groups | kept=2556
[global-fit] processed 3000/7213 groups | kept=2791
[global-fit] processed 3250/7213 groups | kept=3022
[global-fit] processed 3500/7213 groups | kept=3265
[global-fit] processed 3750/7213 groups | kept=3506
[global-fit] processed 4000/7213 groups | kept=3743
[global-fit] processed 4250/7213 groups | kept=3976
[global-fit] processed 4500/7213 groups | kept=4212
[global-fit] processed 47

,session_id,unit,brain_region,interval_name,interval_label,n_trials,mean_count,var_count,overdispersion_index,family_selected,poisson_cv_loglik,negbin_cv_loglik,full_fde_cv,full_fde_train,alpha_nb,cue_coef,cue_p,cue_delta_fde,choice_coef,choice_p,choice_delta_fde,outcome_coef,outcome_p,outcome_delta_fde,dominant_feature,dominant_delta_fde,cue_q,cue_sig,choice_q,choice_sig,outcome_q,outcome_sig,preferred_feature
0,2024-11-14_16-40,Unit0001,CA1,R1_entry_interval,R1 entry,26,3.500000,6.820000,1.948571,negbin,-6.683519e+01,-60.318044,0.010035,0.389193,0.271020,-0.176169,0.607089,-0.023000,-0.229175,0.682746,-0.040612,-0.670588,0.103571,0.052676,outcome,0.052676,1.000000,False,1.000000,False,0.873711,False,none
1,2024-11-14_16-40,Unit0002,CA1,R1_entry_interval,R1 entry,26,1.230769,1.064615,0.865000,poisson,-4.330898e+01,-43.308978,-0.464834,0.180502,NaN,-0.418191,0.300041,-0.019231,0.592951,0.349583,-0.060437,-0.153867,0.753696,-0.232174,cue,-0.019231,0.962007,False,0.971796,False,1.000000,False,none
2,2024-11-14_16-40,Unit0003,CA1,R1_entry_interval,R1 entry,26,0.423077,0.573846,1.356364,negbin,-7.832049e+01,-54.538643,-2.300091,0.418319,0.842314,9.395890,0.529468,0.088028,-11.056098,0.144209,-0.323202,9.306526,0.573581,-0.050409,cue,0.088028,1.000000,False,0.837630,False,1.000000,False,none
3,2024-11-14_16-40,Unit0005,DG,R1_entry_interval,R1 entry,26,1.346154,1.515385,1.125714,negbin,-6.267989e+01,-60.532297,-1.165781,0.302136,0.093388,-0.400235,0.395620,-0.576827,-0.838487,0.209759,-0.204094,-0.456874,0.408455,-0.090786,outcome,-0.090786,0.982540,False,0.879403,False,0.975608,False,none
4,2024-11-14_16-40,Unit0007,DG,R1_entry_interval,R1 entry,26,0.115385,0.106154,0.920000,poisson,-4.430296e+06,-inf,-634140.250736,0.598324,NaN,-3.410654,0.999987,-603848.663291,3.656346,0.999976,-628770.997795,23.333533,0.166451,698227.144582,outcome,698227.144582,1.000000,False,1.000000,False,0.912819,False,none
5,2024-11-14_16-40,Unit0009,DG,R1_entry_interval,R1 entry,26,0.423077,0.413846,0.978182,poisson,-6.415325e+01,-inf,-3.218547,0.172979,NaN,-0.093901,0.899195,-1.589403,0.240996,0.834829,-1.608285,-0.368885,0.678825,-1.493795,outcome,-1.493795,1.000000,False,1.000000,False,1.000000,False,none
6,2024-11-14_16-40,Unit0010,DG,R1_entry_interval,R1 entry,26,15.576923,36.333846,2.332543,negbin,-1.033772e+02,-92.262895,-0.297781,0.324688,0.085546,-0.069234,0.684130,-0.144249,-0.244043,0.355518,-0.039889,-0.021563,0.915741,-0.032638,outcome,-0.032638,1.000000,False,0.972007,False,1.000000,False,none
7,2024-11-14_16-40,Unit0012,DG,R1_entry_interval,R1 entry,26,5.730769,24.524615,4.279463,negbin,-1.447860e+02,-90.956601,-0.998755,0.171912,0.572255,-0.655750,0.106196,0.088316,0.034976,0.951933,-0.432832,-0.358920,0.475129,-0.109225,cue,0.088316,0.858076,False,1.000000,False,0.991963,False,none
8,2024-11-14_16-40,Unit0013,DG,R1_entry_interval,R1 entry,26,0.269231,0.364615,1.354286,negbin,-2.260808e+02,-80.662142,-6.685381,0.494757,1.315918,0.125189,0.999990,0.001019,-22.699662,0.052730,-0.808460,-1.769693,0.999979,-0.021180,cue,0.001019,1.000000,False,0.759874,False,1.000000,False,none
9,2024-11-14_16-40,Unit0015,DG,R1_entry_interval,R1 entry,26,79.230769,265.304615,3.348505,negbin,-1.595552e+02,-120.272446,-0.692396,0.212689,0.029641,-0.144578,0.108625,0.299575,-0.088946,0.523252,-0.043859,-0.122167,0.261624,-0.274590,cue,0.299575,0.860487,False,0.975430,False,0.926650,False,none


In [5]:
family_summary = (
    glm_results.groupby("family_selected", as_index=False)
    .size()
    .rename(columns={"size": "n_fits"})
)
family_summary["fraction"] = family_summary["n_fits"] / family_summary["n_fits"].sum()

fig_family = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Mean-variance structure of interval spike counts", "Held-out family selection"),
    horizontal_spacing=0.12,
)

scatter_df = glm_results.copy()
scatter_df["log_mean_count"] = np.log10(np.clip(scatter_df["mean_count"], 1e-3, None))
scatter_df["log_var_count"] = np.log10(np.clip(scatter_df["var_count"], 1e-3, None))

for family_name, family_df in scatter_df.groupby("family_selected", sort=False):
    fig_family.add_trace(
        go.Scatter(
            x=family_df["mean_count"],
            y=family_df["var_count"],
            mode="markers",
            name=family_name,
            marker=dict(size=7, opacity=0.65),
            customdata=np.stack(
                [
                    family_df["session_id"],
                    family_df["unit"],
                    family_df["interval_label"],
                    family_df["full_fde_cv"].round(3),
                ],
                axis=1,
            ),
            hovertemplate=(
                "Session=%{customdata[0]}<br>"
                "Unit=%{customdata[1]}<br>"
                "Interval=%{customdata[2]}<br>"
                "mean=%{x:.2f}<br>"
                "var=%{y:.2f}<br>"
                "full FDE=%{customdata[3]}<extra></extra>"
            ),
            showlegend=True,
        ),
        row=1,
        col=1,
    )

max_diag = float(np.nanmax(scatter_df[["mean_count", "var_count"]].to_numpy())) if len(scatter_df) else 1.0
diag_x = np.linspace(1e-3, max(max_diag, 1.0), 200)
fig_family.add_trace(
    go.Scatter(
        x=diag_x,
        y=diag_x,
        mode="lines",
        line=dict(color="black", dash="dash"),
        name="Var = Mean",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig_family.add_trace(
    go.Bar(
        x=family_summary["family_selected"],
        y=family_summary["fraction"],
        text=family_summary["fraction"].map(lambda x: f"{100*x:.1f}%"),
        textposition="outside",
        marker_color=["#33658A" if fam == "poisson" else "#BC4B51" for fam in family_summary["family_selected"]],
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig_family.update_xaxes(type="log", title_text="Mean spike count / interval", row=1, col=1)
fig_family.update_yaxes(type="log", title_text="Variance spike count / interval", row=1, col=1)
fig_family.update_xaxes(title_text="Selected family", row=1, col=2)
fig_family.update_yaxes(title_text="Fraction of fits", tickformat=".0%", row=1, col=2)
fig_family.update_layout(height=460, width=1100, title_text="Count-model diagnostics")
fig_family.show()

global_summary = (
    feature_results.groupby(["interval_label", "feature"], as_index=False)
    .agg(
        frac_significant=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
        median_full_fde=("full_fde_cv", "median"),
        n_fits=("feature", "size"),
        n_significant=("is_significant", "sum"),
    )
)

heatmap_y = [INTERVAL_LABELS[name] for name in MAIN_INTERVALS]
heatmap_x = PRIMARY_FEATURES
pivot_frac = (
    global_summary.pivot(index="interval_label", columns="feature", values="frac_significant")
    .reindex(index=heatmap_y, columns=heatmap_x)
)
pivot_delta = (
    global_summary.pivot(index="interval_label", columns="feature", values="median_delta_fde")
    .reindex(index=heatmap_y, columns=heatmap_x)
)
pivot_full = (
    global_summary.pivot(index="interval_label", columns="feature", values="median_full_fde")
    .reindex(index=heatmap_y, columns=heatmap_x)
)
pivot_n = (
    global_summary.pivot(index="interval_label", columns="feature", values="n_fits")
    .reindex(index=heatmap_y, columns=heatmap_x)
)

customdata = np.dstack(
    [
        np.round(pivot_delta.to_numpy(dtype=float), 4),
        np.round(pivot_full.to_numpy(dtype=float), 4),
        pivot_n.to_numpy(dtype=float),
    ]
)

fig_global = go.Figure(
    data=[
        go.Heatmap(
            z=pivot_frac.to_numpy(dtype=float),
            x=heatmap_x,
            y=heatmap_y,
            colorscale="YlOrRd",
            zmin=0.0,
            zmax=max(0.05, float(np.nanmax(pivot_frac.to_numpy(dtype=float)))),
            customdata=customdata,
            text=np.vectorize(lambda x: f"{100*x:.1f}%" if np.isfinite(x) else "NA")(pivot_frac.to_numpy(dtype=float)),
            texttemplate="%{text}",
            hovertemplate=(
                "Interval=%{y}<br>"
                "Feature=%{x}<br>"
                "Frac significant=%{z:.3f}<br>"
                "Median delta FDE=%{customdata[0]:.4f}<br>"
                "Median full FDE=%{customdata[1]:.4f}<br>"
                "n fits=%{customdata[2]:.0f}<extra></extra>"
            ),
            colorbar=dict(title="Fraction significant"),
        )
    ]
)
fig_global.update_layout(
    title="Global screen: which task features dominate which event windows?",
    width=920,
    height=520,
)
fig_global.show()

display(global_summary.sort_values(["frac_significant", "median_delta_fde"], ascending=[False, False]).head(18))


,interval_label,feature,frac_significant,median_delta_fde,median_full_fde,n_fits,n_significant
2,Cue entry,outcome,0.001279,-0.017559,-0.170662,1564,2
11,R2 entry,outcome,0.001263,-0.019196,-0.196685,1583,2
4,Cue exit,cue,0.000665,-0.014966,-0.165398,1503,1
5,Cue exit,outcome,0.000665,-0.021313,-0.165398,1503,1
1,Cue entry,cue,0.000639,-0.013431,-0.170662,1564,1
0,Cue entry,choice,0.000639,-0.018763,-0.170662,1564,1
8,R1 entry,outcome,0.000635,-0.014902,-0.166188,1575,1
7,R1 entry,cue,0.000635,-0.019231,-0.166188,1575,1
9,R2 entry,choice,0.000632,-0.020272,-0.196685,1583,1
10,R2 entry,cue,0.000000,-0.017365,-0.196685,1583,0


In [6]:
session_order = pd.DataFrame({"session_id": feature_results["session_id"].drop_duplicates()})
session_order["session_dt"] = pd.to_datetime(session_order["session_id"], format="%Y-%m-%d_%H-%M", errors="coerce")
if session_order["session_dt"].notna().any():
    session_order = session_order.sort_values(["session_dt", "session_id"])
else:
    session_order = session_order.sort_values("session_id")
ordered_sessions = session_order["session_id"].tolist()

session_unit_feature = (
    feature_results.groupby(["session_id", "unit", "feature"], as_index=False)
    .agg(
        sig_any=("is_significant", "max"),
        max_delta_fde=("delta_fde", "max"),
        median_full_fde=("full_fde_cv", "median"),
    )
)
session_feature_summary = (
    session_unit_feature.groupby(["session_id", "feature"], as_index=False)
    .agg(
        frac_units_sig=("sig_any", "mean"),
        median_max_delta_fde=("max_delta_fde", "median"),
        median_full_fde=("median_full_fde", "median"),
        n_units=("unit", "nunique"),
    )
)
session_feature_summary["session_id"] = pd.Categorical(
    session_feature_summary["session_id"],
    categories=ordered_sessions,
    ordered=True,
)
session_feature_summary = session_feature_summary.sort_values(["session_id", "feature"])

heat_frac = (
    session_feature_summary.pivot(index="feature", columns="session_id", values="frac_units_sig")
    .reindex(index=PRIMARY_FEATURES, columns=ordered_sessions)
)
heat_delta = (
    session_feature_summary.pivot(index="feature", columns="session_id", values="median_max_delta_fde")
    .reindex(index=PRIMARY_FEATURES, columns=ordered_sessions)
)
heat_n = (
    session_feature_summary.pivot(index="feature", columns="session_id", values="n_units")
    .reindex(index=PRIMARY_FEATURES, columns=ordered_sessions)
)

session_custom = np.dstack(
    [
        np.round(heat_delta.to_numpy(dtype=float), 4),
        heat_n.to_numpy(dtype=float),
    ]
)

fig_session = go.Figure(
    data=[
        go.Heatmap(
            z=heat_frac.to_numpy(dtype=float),
            x=ordered_sessions,
            y=PRIMARY_FEATURES,
            colorscale="Blues",
            zmin=0.0,
            zmax=max(0.05, float(np.nanmax(heat_frac.to_numpy(dtype=float)))),
            customdata=session_custom,
            hovertemplate=(
                "Session=%{x}<br>"
                "Feature=%{y}<br>"
                "Frac units significant=%{z:.3f}<br>"
                "Median max delta FDE=%{customdata[0]:.4f}<br>"
                "n units=%{customdata[1]:.0f}<extra></extra>"
            ),
            colorbar=dict(title="Fraction units significant"),
        )
    ]
)
fig_session.update_layout(
    title="Session-wise tuning: prevalence of cue / choice / outcome encoding",
    width=max(1050, 42 * len(ordered_sessions)),
    height=340,
)
fig_session.update_xaxes(tickangle=-45)
fig_session.show()

display(session_feature_summary.head(18))


,session_id,feature,frac_units_sig,median_max_delta_fde,median_full_fde,n_units
0,2024-11-14_16-40,choice,0.000000,0.017002,-0.638587,63
1,2024-11-14_16-40,cue,0.000000,0.012682,-0.638587,63
2,2024-11-14_16-40,outcome,0.000000,0.007994,-0.638587,63
3,2024-11-15_15-48,choice,0.000000,0.005750,-0.276824,67
4,2024-11-15_15-48,cue,0.000000,-0.001285,-0.276824,67
5,2024-11-15_15-48,outcome,0.029851,0.012315,-0.276824,67
6,2024-11-21_17-22,choice,0.015152,0.931663,-1.360229,66
7,2024-11-21_17-22,cue,0.000000,0.372435,-1.360229,66
8,2024-11-21_17-22,outcome,0.015152,0.286471,-1.360229,66
9,2024-11-25_16-25,choice,0.000000,0.006661,-3.295311,63


In [7]:
def fit_phase_delta(group_df: pd.DataFrame, feature_name: str, family_name: str) -> list[dict]:
    feature_col = FEATURE_SPECS[feature_name]["column"]
    trial_df = group_df.sort_values("trial_number").copy()
    if trial_df["trial_number"].nunique() < (MIN_PHASE_TRIALS * 3):
        return []

    try:
        phase_index = pd.qcut(
            trial_df["trial_number"].rank(method="first"),
            q=3,
            labels=["early", "middle", "late"],
            duplicates="drop",
        )
    except Exception:
        return []

    trial_df["phase"] = phase_index.astype(str)
    rows = []
    for phase_name, phase_df in trial_df.groupby("phase", sort=False):
        if len(phase_df) < MIN_PHASE_TRIALS:
            continue
        predictors = _available_predictors(phase_df)
        if feature_col not in predictors:
            continue
        try:
            full_res, full_x, alpha = _fit_glm(phase_df, predictors, family_name, ref_df=phase_df)
            null_res, null_x, _ = _fit_glm(phase_df, [], family_name, ref_df=phase_df)
            reduced_predictors = [col for col in predictors if col != feature_col]
            reduced_res, red_x, _ = _fit_glm(phase_df, reduced_predictors, family_name, ref_df=phase_df)
        except Exception:
            continue

        phase_offset = np.log(np.clip(phase_df["n_bins_present"].to_numpy(dtype=float), 1.0, None))
        y_phase = phase_df["spike_count"].to_numpy(dtype=int)
        mu_full = np.clip(full_res.predict(full_x, offset=phase_offset), 1e-9, None)
        mu_null = np.clip(null_res.predict(null_x, offset=phase_offset), 1e-9, None)
        mu_reduced = np.clip(reduced_res.predict(red_x, offset=phase_offset), 1e-9, None)

        dev_full = _heldout_deviance(y_phase, mu_full, family_name, alpha)
        dev_null = _heldout_deviance(y_phase, mu_null, family_name, alpha)
        dev_reduced = _heldout_deviance(y_phase, mu_reduced, family_name, alpha)
        full_fde = np.nan if dev_null <= 0 else 1.0 - (dev_full / dev_null)
        red_fde = np.nan if dev_null <= 0 else 1.0 - (dev_reduced / dev_null)
        rows.append(
            {
                "phase": phase_name,
                "phase_order": {"early": 0, "middle": 1, "late": 2}.get(phase_name, np.nan),
                "phase_delta_fde": full_fde - red_fde if np.isfinite(full_fde) and np.isfinite(red_fde) else np.nan,
                "phase_full_fde": full_fde,
                "n_trials_phase": int(phase_df["trial_id"].nunique()),
            }
        )
    return rows


def fit_interaction(group_df: pd.DataFrame, feature_name: str, family_name: str) -> dict | None:
    feature_col = FEATURE_SPECS[feature_name]["column"]
    work_df = group_df.sort_values("trial_number").copy()
    predictors = _available_predictors(work_df)
    if feature_col not in predictors:
        return None

    interaction_col = f"{feature_col}_x_trial"
    work_df[interaction_col] = work_df[feature_col] * work_df["trial_number_z"]
    try:
        base_res, _, _ = _fit_glm(work_df, predictors, family_name, ref_df=work_df)
        inter_res, _, _ = _fit_glm(work_df, predictors + [interaction_col], family_name, ref_df=work_df)
    except Exception:
        return None

    return {
        "interaction_coef": inter_res.params.get(interaction_col, np.nan),
        "interaction_p": _lr_pvalue(inter_res, base_res),
    }


top_pairs = (
    feature_results.dropna(subset=["delta_fde"])
    .groupby(["interval_name", "interval_label", "feature"], as_index=False)
    .agg(
        frac_significant=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
    )
    .sort_values(["feature", "frac_significant", "median_delta_fde"], ascending=[True, False, False])
    .groupby("feature", as_index=False)
    .head(TOP_TRIALWISE_PAIRS_PER_FEATURE)
    .reset_index(drop=True)
)

trialwise_rows = []
interaction_rows = []

for pair in top_pairs.itertuples(index=False):
    pair_feature = pair.feature
    pair_interval = pair.interval_name
    pair_interval_label = pair.interval_label
    pair_results = feature_results[
        (feature_results["feature"] == pair_feature)
        & (feature_results["interval_name"] == pair_interval)
        & (feature_results["is_significant"])
    ].copy()

    for res_row in pair_results.itertuples(index=False):
        group_df = unit_window_table[
            (unit_window_table["session_id"] == res_row.session_id)
            & (unit_window_table["unit"] == res_row.unit)
            & (unit_window_table["interval_name"] == pair_interval)
        ].copy()
        if group_df.empty:
            continue

        interaction = fit_interaction(group_df, pair_feature, res_row.family_selected)
        if interaction is not None:
            interaction_rows.append(
                {
                    "session_id": res_row.session_id,
                    "unit": res_row.unit,
                    "interval_name": pair_interval,
                    "interval_label": pair_interval_label,
                    "feature": pair_feature,
                    **interaction,
                }
            )

        for phase_row in fit_phase_delta(group_df, pair_feature, res_row.family_selected):
            trialwise_rows.append(
                {
                    "session_id": res_row.session_id,
                    "unit": res_row.unit,
                    "interval_name": pair_interval,
                    "interval_label": pair_interval_label,
                    "feature": pair_feature,
                    **phase_row,
                }
            )

trialwise_phase_df = pd.DataFrame(trialwise_rows)
interaction_df = pd.DataFrame(interaction_rows)

if not interaction_df.empty:
    interaction_df["interaction_q"] = np.nan
    for feature_name in interaction_df["feature"].dropna().unique():
        mask = interaction_df["feature"] == feature_name
        interaction_df.loc[mask, "interaction_q"] = multipletests(
            interaction_df.loc[mask, "interaction_p"], method="fdr_bh"
        )[1]
    interaction_df["interaction_sig"] = interaction_df["interaction_q"] < FDR_ALPHA

phase_summary = (
    trialwise_phase_df.groupby(["feature", "interval_label", "phase", "phase_order"], as_index=False)
    .agg(
        median_phase_delta_fde=("phase_delta_fde", "median"),
        lo_phase_delta_fde=("phase_delta_fde", lambda s: np.nanpercentile(s, 25) if len(s) else np.nan),
        hi_phase_delta_fde=("phase_delta_fde", lambda s: np.nanpercentile(s, 75) if len(s) else np.nan),
        n_units=("unit", "nunique"),
    )
    .sort_values(["feature", "phase_order"])
)

interaction_summary = (
    interaction_df.groupby(["feature", "interval_label"], as_index=False)
    .agg(
        frac_sig_interaction=("interaction_sig", "mean"),
        median_interaction_coef=("interaction_coef", "median"),
        n_units=("unit", "nunique"),
    )
    if not interaction_df.empty
    else pd.DataFrame(columns=["feature", "interval_label", "frac_sig_interaction", "median_interaction_coef", "n_units"])
)

fig_trialwise = go.Figure()
color_map = {"cue": "#33658A", "choice": "#BC4B51", "outcome": "#758E4F"}

for (feature_name, interval_label), sub in phase_summary.groupby(["feature", "interval_label"], sort=False):
    sub = sub.sort_values("phase_order")
    interaction_info = interaction_summary[
        (interaction_summary["feature"] == feature_name)
        & (interaction_summary["interval_label"] == interval_label)
    ]
    if interaction_info.empty:
        hover_suffix = "sig interaction fraction=NA"
    else:
        hover_suffix = f"sig interaction fraction={float(interaction_info['frac_sig_interaction'].iat[0]):.3f}"

    fig_trialwise.add_trace(
        go.Scatter(
            x=sub["phase"],
            y=sub["median_phase_delta_fde"],
            mode="lines+markers",
            name=f"{feature_name} | {interval_label}",
            line=dict(color=color_map.get(feature_name, "#444"), width=3),
            marker=dict(size=8),
            error_y=dict(
                type="data",
                symmetric=False,
                array=(sub["hi_phase_delta_fde"] - sub["median_phase_delta_fde"]).clip(lower=0),
                arrayminus=(sub["median_phase_delta_fde"] - sub["lo_phase_delta_fde"]).clip(lower=0),
            ),
            customdata=np.stack([sub["n_units"]], axis=1),
            hovertemplate=(
                "Phase=%{x}<br>"
                "Median delta FDE=%{y:.4f}<br>"
                "n units=%{customdata[0]:.0f}<br>"
                + hover_suffix
                + "<extra></extra>"
            ),
        )
    )

fig_trialwise.update_layout(
    title="Trial-wise refinement: strongest interval-feature pairs across early / middle / late trials",
    width=980,
    height=480,
    yaxis_title="Median delta FDE",
    xaxis_title="Within-session trial phase",
)
fig_trialwise.show()

display(phase_summary)
display(interaction_summary)


,feature,interval_label,phase,phase_order,median_phase_delta_fde,lo_phase_delta_fde,hi_phase_delta_fde,n_units
0,choice,Cue entry,early,0,-3.330669e-16,-3.330669e-16,-3.330669e-16,1
2,choice,Cue entry,middle,1,2.220446e-16,2.220446e-16,2.220446e-16,1
1,choice,Cue entry,late,2,1.956948e-01,1.956948e-01,1.956948e-01,1
3,cue,Cue exit,early,0,0.000000e+00,0.000000e+00,0.000000e+00,1
4,cue,Cue exit,late,2,2.765232e-02,2.765232e-02,2.765232e-02,1
5,outcome,Cue entry,early,0,1.704922e-01,1.704922e-01,1.704922e-01,1
7,outcome,Cue entry,middle,1,8.813251e-02,8.813251e-02,8.813251e-02,1
6,outcome,Cue entry,late,2,2.357841e-01,2.357841e-01,2.357841e-01,1


,feature,interval_label,frac_sig_interaction,median_interaction_coef,n_units
0,choice,Cue entry,0.0,-0.031582,1
1,cue,Cue exit,0.0,28.114287,1
2,outcome,Cue entry,0.5,-16.306080,2


In [8]:
def refit_group_with_predictions(group_df: pd.DataFrame, family_name: str):
    predictors = _available_predictors(group_df)
    full_res, full_x, alpha = _fit_glm(group_df, predictors, family_name, ref_df=group_df)
    full_offset = np.log(np.clip(group_df["n_bins_present"].to_numpy(dtype=float), 1.0, None))
    pred = np.clip(full_res.predict(full_x, offset=full_offset), 1e-9, None)
    out = group_df.copy()
    out["predicted_count"] = pred
    return out, full_res, alpha


exemplar_rows = []
for feature_name in PRIMARY_FEATURES:
    sub = feature_results[
        (feature_results["feature"] == feature_name)
        & (feature_results["is_significant"])
    ].copy()
    if sub.empty:
        continue
    sub = sub.sort_values(["delta_fde", "full_fde_cv", "n_trials"], ascending=[False, False, False])
    exemplar_rows.append(sub.iloc[0])

exemplar_df = pd.DataFrame(exemplar_rows)

fig_ex = make_subplots(
    rows=max(len(exemplar_df), 1),
    cols=2,
    subplot_titles=[
        f"{row.feature.title()} exemplar: {row.interval_label} | {row.unit} | {row.session_id}"
        for row in exemplar_df.itertuples(index=False)
        for _ in range(2)
    ] if len(exemplar_df) else ["No significant exemplars found", ""],
    horizontal_spacing=0.12,
    vertical_spacing=0.12,
)

if exemplar_df.empty:
    fig_ex.add_trace(go.Scatter(x=[0], y=[0], mode="text", text=["No significant cue / choice / outcome exemplars found."]), row=1, col=1)
else:
    for row_idx, ex_row in enumerate(exemplar_df.itertuples(index=False), start=1):
        group_df = unit_window_table[
            (unit_window_table["session_id"] == ex_row.session_id)
            & (unit_window_table["unit"] == ex_row.unit)
            & (unit_window_table["interval_name"] == ex_row.interval_name)
        ].copy()
        fitted_df, _, _ = refit_group_with_predictions(group_df, ex_row.family_selected)
        feature_col = FEATURE_SPECS[ex_row.feature]["column"]
        label0 = FEATURE_SPECS[ex_row.feature]["label0"]
        label1 = FEATURE_SPECS[ex_row.feature]["label1"]

        condition_summary = (
            fitted_df.groupby(feature_col, as_index=False)
            .agg(
                observed_mean=("spike_count", "mean"),
                predicted_mean=("predicted_count", "mean"),
            )
            .sort_values(feature_col)
        )
        if len(condition_summary) == 2:
            condition_summary["condition_label"] = [label0, label1]
        else:
            condition_summary["condition_label"] = condition_summary[feature_col].astype(str)

        fig_ex.add_trace(
            go.Bar(
                x=condition_summary["condition_label"],
                y=condition_summary["observed_mean"],
                name=f"{ex_row.feature} observed",
                marker_color="#7A7A7A",
                opacity=0.75,
                legendgroup=f"{ex_row.feature}_obs",
                showlegend=(row_idx == 1),
            ),
            row=row_idx,
            col=1,
        )
        fig_ex.add_trace(
            go.Scatter(
                x=condition_summary["condition_label"],
                y=condition_summary["predicted_mean"],
                mode="lines+markers",
                name=f"{ex_row.feature} predicted",
                line=dict(color="#BC4B51", width=3),
                marker=dict(size=9),
                legendgroup=f"{ex_row.feature}_pred",
                showlegend=(row_idx == 1),
            ),
            row=row_idx,
            col=1,
        )

        roll_window = max(ROLLING_WINDOW_MIN, int(np.ceil(len(fitted_df) / 8)))
        fitted_df = fitted_df.sort_values("trial_number").copy()
        condition_map = {0.0: label0, 1.0: label1}
        for cond_value, cond_label in condition_map.items():
            cond_df = fitted_df[fitted_df[feature_col] == cond_value].copy()
            if cond_df.empty:
                continue
            cond_df["observed_roll"] = cond_df["spike_count"].rolling(roll_window, min_periods=1).mean()
            cond_df["predicted_roll"] = cond_df["predicted_count"].rolling(roll_window, min_periods=1).mean()

            fig_ex.add_trace(
                go.Scatter(
                    x=cond_df["trial_number"],
                    y=cond_df["observed_roll"],
                    mode="lines",
                    line=dict(color="#7A7A7A", dash="dot"),
                    name=f"{cond_label} observed rolling",
                    legendgroup=f"{ex_row.feature}_{cond_label}_obs",
                    showlegend=False,
                ),
                row=row_idx,
                col=2,
            )
            fig_ex.add_trace(
                go.Scatter(
                    x=cond_df["trial_number"],
                    y=cond_df["predicted_roll"],
                    mode="lines",
                    line=dict(color="#33658A" if cond_value == 0 else "#BC4B51", width=3),
                    name=f"{cond_label} predicted rolling",
                    legendgroup=f"{ex_row.feature}_{cond_label}_pred",
                    showlegend=(row_idx == 1),
                ),
                row=row_idx,
                col=2,
            )

        fig_ex.update_yaxes(title_text="Mean count", row=row_idx, col=1)
        fig_ex.update_yaxes(title_text="Rolling mean count", row=row_idx, col=2)
        fig_ex.update_xaxes(title_text=ex_row.feature.title(), row=row_idx, col=1)
        fig_ex.update_xaxes(title_text="Trial number", row=row_idx, col=2)

fig_ex.update_layout(
    title="Representative single neurons: observed condition means and GLM predictions",
    width=1180,
    height=max(420, 320 * max(len(exemplar_df), 1)),
    barmode="group",
)
fig_ex.show()

display(exemplar_df)


,session_id,unit,brain_region,interval_name,interval_label,family_selected,full_fde_cv,n_trials,feature,delta_fde,p_value,q_value,is_significant,coef
4537,2025-01-15_17-18,Unit0006,DG,cue_exit_interval,Cue exit,poisson,-6.051447e+23,44,cue,7.739650e+24,8.723188e-15,1.983217e-11,True,13.153230
7444,2024-11-21_17-22,Unit0076,InfrL,R2_entry_interval,R2 entry,poisson,3.393994e-01,20,choice,1.078173e+00,1.911746e-05,4.347311e-02,True,-0.645485
18041,2025-01-15_17-18,Unit0020,DG,R2_entry_interval,R2 entry,negbin,-1.123579e+00,44,outcome,8.912188e+01,1.026616e-22,4.669048e-19,True,-1.595501


In [9]:
summary_lines = []

if not family_summary.empty:
    top_family = family_summary.sort_values("fraction", ascending=False).iloc[0]
    summary_lines.append(
        f"- **Count family**: `{top_family['family_selected']}` was selected most often "
        f"({100 * top_family['fraction']:.1f}% of eligible neuron-session-interval fits)."
    )

if not global_summary.empty:
    top_global = global_summary.sort_values(["frac_significant", "median_delta_fde"], ascending=[False, False]).iloc[0]
    summary_lines.append(
        f"- **Strongest coarse effect**: `{top_global['feature']}` was most prominent in "
        f"`{top_global['interval_label']}` with {100 * top_global['frac_significant']:.1f}% significant fits "
        f"and median delta FDE {top_global['median_delta_fde']:.4f}."
    )

if not session_feature_summary.empty:
    top_session = session_feature_summary.sort_values(
        ["frac_units_sig", "median_max_delta_fde"], ascending=[False, False]
    ).iloc[0]
    summary_lines.append(
        f"- **Session-wise tuning**: the clearest session-level prevalence appeared for "
        f"`{top_session['feature']}` in session `{top_session['session_id']}` "
        f"({100 * top_session['frac_units_sig']:.1f}% of units significant)."
    )

if not phase_summary.empty:
    tmp = phase_summary.pivot_table(
        index=["feature", "interval_label"],
        columns="phase",
        values="median_phase_delta_fde",
        aggfunc="first",
    ).reset_index()
    if {"early", "late"}.issubset(tmp.columns):
        tmp["late_minus_early"] = tmp["late"] - tmp["early"]
        top_phase = tmp.reindex(tmp["late_minus_early"].abs().sort_values(ascending=False).index).iloc[0]
        summary_lines.append(
            f"- **Trial-wise refinement**: `{top_phase['feature']}` in `{top_phase['interval_label']}` changed most "
            f"across the session (late - early delta FDE = {top_phase['late_minus_early']:.4f})."
        )

if not exemplar_df.empty:
    ex_items = [
        f"`{row.feature}`: {row.unit} in {row.interval_label} ({row.session_id})"
        for row in exemplar_df.itertuples(index=False)
    ]
    summary_lines.append("- **Representative neurons**: " + "; ".join(ex_items) + ".")

if not summary_lines:
    summary_lines.append("- No stable summary lines were generated from the current results.")

display(Markdown("## Concise Findings\n" + "\n".join(summary_lines)))


## Concise Findings
- **Count family**: `negbin` was selected most often (73.9% of eligible neuron-session-interval fits).
- **Strongest coarse effect**: `outcome` was most prominent in `Cue entry` with 0.1% significant fits and median delta FDE -0.0176.
- **Session-wise tuning**: the clearest session-level prevalence appeared for `outcome` in session `2024-11-15_15-48` (3.0% of units significant).
- **Trial-wise refinement**: `choice` in `Cue entry` changed most across the session (late - early delta FDE = 0.1957).
- **Representative neurons**: `cue`: Unit0006 in Cue exit (2025-01-15_17-18); `choice`: Unit0076 in R2 entry (2024-11-21_17-22); `outcome`: Unit0020 in R2 entry (2025-01-15_17-18).